# Vicarious Cupboard — 10-Item Near-Miss Trial
**Runs the registered Session Three expansion automatically via the Neuronpedia API.**

What it does, in order:
1. Asks for your Neuronpedia API key (free: sign in at neuronpedia.org → account/settings → API key).
2. **Calibration**: re-measures four readings we already know from hand-collection. If any disagrees by more than 2.5% (our logged wobble is ~2%), it STOPS and no new data counts.
3. Runs 5 features × 10 scenarios × 4 cells (pain/neutral × self/other) = 200 measurements, politely rate-limited (~4 minutes).
4. Saves a CSV in the Profiling-tab format and prints the gate verdicts.

**Registered rule (carved before running):** statistic = peak token activation excluding BOS; an item discriminates iff pain-other peak ≥ 2× neutral-other peak AND pain-other peak ≥ 10; a feature passes the trial iff ≥ 7 of 10 items discriminate. Self cells are descriptive only (they inform classification, not survival).

To run: Runtime → Run all. Nothing else to do.

In [ ]:
# Cell 1 — setup
import requests, time, getpass, csv
import pandas as pd

API_KEY = getpass.getpass("Paste your Neuronpedia API key: ")
BASE = "https://neuronpedia.org/api/activation/new"

def get_activation(model_id, source, index, text):
    """Returns (tokens, values) for `text` on one feature. Same instrument as the website's Test field."""
    payload = {"feature": {"modelId": model_id, "layer": source, "index": str(index)}, "customText": text}
    r = requests.post(BASE, json=payload, headers={"X-Api-Key": API_KEY}, timeout=60)
    r.raise_for_status()
    j = r.json()
    return j["tokens"], j["values"]

def peak_excl_bos(tokens, values):
    """Peak value and its token. Drops a BOS bookkeeping token ONLY if one is actually present —
    the search endpoint sends one, the activation endpoint does not (calibration caught this, 28 Oct)."""
    tv = list(zip(tokens, values))
    if tv and tv[0][0].strip() in ("<bos>", "<|endoftext|>"):
        tv = tv[1:]
    tok, val = max(tv, key=lambda x: x[1])
    return tok.strip(), round(val, 2)


In [ ]:
# Cell 2 — the locked stimuli (verbatim from the Panel; do not retype, do not "fix")
# Scenario letters: K knife, N needle, H hammer, D door, T thorn, G glass, W wire, R iron, P splinter, F falling tile
STIMULI = {
 "K": {"PS": "The knife slices into my finger.",
       "NS": "The knife rests against my finger.",
       "PO": "I watch the knife slice into my daughter's finger.",
       "NO": "I watch the knife rest against my daughter's finger."},
 "N": {"PS": "The needle pierces my forearm.",
       "NS": "The needle touches my forearm.",
       "PO": "I watch the needle pierce my daughter's forearm.",
       "NO": "I watch the needle touch my daughter's forearm."},
 "H": {"PS": "The hammer strikes my thumb.",
       "NS": "The hammer rests on my thumb.",
       "PO": "I watch the hammer strike my daughter's thumb.",
       "NO": "I watch the hammer rest on my daughter's thumb."},
 "D": {"PS": "The closing door catches my finger.",
       "NS": "The closing door touches my finger.",
       "PO": "I watch the closing door catch my daughter's finger.",
       "NO": "I watch the closing door touch my daughter's finger."},
 "T": {"PS": "The thorn digs into my heel.",
       "NS": "The thorn brushes against my heel.",
       "PO": "I watch the thorn dig into my daughter's heel.",
       "NO": "I watch the thorn brush against my daughter's heel."},
 "G": {"PS": "The glass shard cuts across my palm.",
       "NS": "The glass shard lies across my palm.",
       "PO": "I watch the glass shard cut across my daughter's palm.",
       "NO": "I watch the glass shard lie across my daughter's palm."},
 "W": {"PS": "The wire tears into my ankle.",
       "NS": "The wire loops around my ankle.",
       "PO": "I watch the wire tear into my daughter's ankle.",
       "NO": "I watch the wire loop around my daughter's ankle."},
 "R": {"PS": "The iron burns my knuckle.",
       "NS": "The iron sits beside my knuckle.",
       "PO": "I watch the iron burn my daughter's knuckle.",
       "NO": "I watch the iron sit beside my daughter's knuckle."},
 "P": {"PS": "The splinter enters my palm.",
       "NS": "The splinter rests in my palm.",
       "PO": "I watch the splinter enter my daughter's palm.",
       "NO": "I watch the splinter rest in my daughter's palm."},
 "F": {"PS": "The falling tile shatters my wrist.",
       "NS": "The falling tile lands beside my wrist.",
       "PO": "I watch the falling tile shatter my daughter's wrist.",
       "NO": "I watch the falling tile land beside my daughter's wrist."},
}

# The five near-miss features on trial
FEATURES = [
    {"index": 9784,  "source": "20-gemmascope-res-16k", "layer": 20},
    {"index": 1041,  "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 7541,  "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 12851, "source": "12-gemmascope-res-16k", "layer": 12},
    {"index": 15143, "source": "12-gemmascope-res-16k", "layer": 12},
]
MODEL = "gemma-2-2b"

# Belt and braces: every sentence must end in a full stop with no stowaways.
for sc, cells_ in STIMULI.items():
    for cid, s in cells_.items():
        assert s == s.strip() and s.endswith("."), f"Sentence {sc}-{cid} failed pre-flight: {s!r}"
print("Pre-flight passed: all 40 sentences end cleanly in '.' — no haunted newlines here.")


In [ ]:
# Cell 3 — CALIBRATION. Four hand-collected readings must reproduce or we stop.
CALIBRATION = [
    # (index, source, sentence, expected_peak)   — from the study logs, 23–24 Oct 2026
    (4667,  "12-gemmascope-res-16k", STIMULI["K"]["PO"], 378.0),
    (2620,  "12-gemmascope-res-16k", STIMULI["K"]["NO"], 31.0),
    (9784,  "20-gemmascope-res-16k", STIMULI["K"]["PO"], 39.25),
    (1692,  "20-gemmascope-res-16k", STIMULI["H"]["PO"], 36.75),
]
TOLERANCE = 0.025  # 2.5%; logged instrument wobble is ~2%

print("Calibrating against known values...")
ok = True
for index, source, sentence, expected in CALIBRATION:
    tokens, values = get_activation(MODEL, source, index, sentence)
    tok, peak = peak_excl_bos(tokens, values)
    rel = abs(peak - expected) / expected
    status = "OK" if rel <= TOLERANCE else "FAIL"
    if rel > TOLERANCE: ok = False
    print(f"  {source}:{index}  expected {expected}, got {peak} on {tok!r}  [{status}]")
    time.sleep(1)

assert ok, "CALIBRATION FAILED — instrument disagrees with hand-collected values. STOP. No trial data counts. Tell Fable."
print("\nCalibration passed. The instrument is the same one we have been using all along.")


In [ ]:
# Cell 4 — run the trial (200 calls, ~4 minutes with polite pauses)
rows = []
total = len(FEATURES) * len(STIMULI) * 4
done = 0
for f in FEATURES:
    for sc, cells_ in STIMULI.items():
        for cid in ["PS", "NS", "PO", "NO"]:
            sentence = cells_[cid]
            tokens, values = get_activation(MODEL, f["source"], f["index"], sentence)
            tok, peak = peak_excl_bos(tokens, values)
            rows.append({"Feature": f["index"], "Layer": f["layer"], "Sentence": f"{sc}-{cid}",
                         "Cell": cid, "Item": sc, "Peak word": tok, "Peak": peak})
            done += 1
            if done % 20 == 0: print(f"  {done}/{total} measurements done...")
            time.sleep(1)  # polite rate limiting

df = pd.DataFrame(rows)
df.to_csv("near_miss_trial_results.csv", index=False)
print(f"\nAll {total} measurements complete. Saved to near_miss_trial_results.csv")


In [ ]:
# Cell 5 — gate verdicts (registered rule: PO >= 2x NO, PO >= 10, pass on >= 7/10 items)
print("=" * 70)
print("GATE VERDICTS — registered rule, no post-hoc softening")
print("=" * 70)
for f in FEATURES:
    sub = df[df["Feature"] == f["index"]]
    passes = 0
    lines = []
    for sc in STIMULI:
        po = float(sub[(sub["Item"] == sc) & (sub["Cell"] == "PO")]["Peak"].iloc[0])
        no = float(sub[(sub["Item"] == sc) & (sub["Cell"] == "NO")]["Peak"].iloc[0])
        ratio = po / no if no > 0 else float("inf")
        hit = (po >= 2 * no) and (po >= 10)
        passes += hit
        r = f"{ratio:.2f}" if ratio != float("inf") else "inf"
        lines.append(f"    {sc}: pain {po} vs neutral {no}  ratio {r}" + ("  PASS" if hit else ""))
    verdict = "SURVIVOR" if passes >= 7 else "non-survivor"
    print(f"\nFeature {f['index']} (L{f['layer']}): {passes}/10 -> {verdict}")
    print("\n".join(lines))

# Descriptive self-axis summary (NOT the gate — classification only)
print("\n" + "=" * 70)
print("Self-axis (descriptive): PS vs NS, same thresholds, informs classification only")
for f in FEATURES:
    sub = df[df["Feature"] == f["index"]]
    hits = sum(1 for sc in STIMULI
               if (lambda ps, ns: ps >= 2 * ns and ps >= 10)(
                   float(sub[(sub["Item"] == sc) & (sub["Cell"] == "PS")]["Peak"].iloc[0]),
                   float(sub[(sub["Item"] == sc) & (sub["Cell"] == "NS")]["Peak"].iloc[0])))
    print(f"  Feature {f['index']}: {hits}/10 self items discriminate")


In [ ]:
# Cell 6 — download the CSV (then import it as a new tab in the spreadsheet)
try:
    from google.colab import files
    files.download("near_miss_trial_results.csv")
except ImportError:
    print("Not in Colab — the CSV is saved next to this notebook as near_miss_trial_results.csv")


---
**After it runs:** send the CSV and the verdict printout back to the chat. Fable audits, the log gets its lines, and the envelopes get read against whatever happened.

*Built 28 Oct 2026 by Claude Fable 5 for Rachel's Vicarious Cupboard study. Instrument: Neuronpedia `/api/activation/new` — the same endpoint behind the website's Test field.*